# qaari-eval — Colab benchmark: studio ijaazah vs. Taraweeh imams

Scores Sheikh Al-Hussary (studio) against Sheikh Yasser Al-Dosari (Taraweeh) — or any reciters
from `benchmarks/roster.py` — on the 59-ayah strategic verse set that exercises every Tajweed
rule, with and without the Taraweeh adapter. Runtime → Change runtime type → **T4 GPU**.

In [ ]:
!git clone https://github.com/akadaan310/muqri.git qaari-eval
%cd qaari-eval
!pip install -q -r requirements-ml.txt nara_wpe

In [ ]:
RECITERS = "Husary_128kbps Yasser_Ad-Dussary_128kbps Nasser_Alqatami_128kbps Minshawy_Murattal_128kbps"
!python benchmarks/run_benchmark.py --reciters {RECITERS} --verses strategic --output benchmarks/results/colab_runs.jsonl
!python benchmarks/summarize.py --runs benchmarks/results/colab_runs.jsonl --no-index

In [ ]:
import json

import matplotlib.pyplot as plt
import pandas as pd

s = json.load(open("benchmarks/results/summary.json"))
rows = []
for r in s["reciters"]:
    for mode, m in r["modes"].items():
        cats = {f"cat:{k}": v for k, v in m["categories"].items()}
        rows.append({"reciter": r["name"], "set": r["category"], "mode": mode, "perfection": m["perfection"],
                     "sifaat": m["sifaat"], "timing_fails": m["timing_fails"], **cats})
df = pd.DataFrame(rows)
display(df[["reciter", "set", "mode", "perfection", "sifaat", "timing_fails"]])
pivot = df[df["mode"] == "studio"].set_index("reciter")[[c for c in df if c.startswith("cat:")]]
pivot.T.plot(kind="bar", figsize=(12, 4), title="Per-family score (studio mode)")
plt.ylabel("score")
plt.show()

In [ ]:
# Pass rate per rule for each reciter (studio mode)
pr = pd.DataFrame(s["pass_rate_by_rule_studio_mode"]).sort_index()
display(pr.style.background_gradient(axis=None, cmap="RdYlGn"))

In [ ]:
# Analyse your own recitation against a benchmark reciter
# !python main.py analyze --audio my_fatiha.wav --surah 1 --tareeq shatibiyyah --mode taraweeh_adapted --benchmark dosari